<a href="https://colab.research.google.com/github/SAIKUNDAN1/Capable-Project/blob/main/travel_concierge_rag_core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Force reinstall and install the core Track A stack
!pip uninstall -y langchain langchain-google-genai langchain-community google-generativeai pydantic
!pip install -q langchain==0.1.20 langchain-google-genai google-generativeai pypdf faiss-cpu langchain-community pydantic>=2

import os
from google.colab import userdata

# Secret Management using Colab's built-in tool
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# Professional directory structure
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print("✅ Environment ready. Upload your travel PDFs to the 'data/raw' folder in the sidebar.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.3 which is incompatible.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.1.53 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.
langgraph-checkpoint 4.0.2 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
✅ Environment ready. Upload your travel PDFs to the 'data/raw' folder in the sidebar.


In [2]:
# Move user-provided PDF files to the 'data/raw' directory
import shutil

source_files = [
    "/content/Hotel_Booking_Confirmation.pdf",
    "/content/Hyderabad_Travel_Brochure (1).pdf"
]

destination_dir = "./data/raw"

for file_path in source_files:
    if os.path.exists(file_path):
        shutil.move(file_path, destination_dir)
        print(f"Moved {os.path.basename(file_path)} to {destination_dir}")
    else:
        print(f"Warning: File not found at {file_path}")

print("\n--- Files in data/raw after moving ---")
print(os.listdir(destination_dir))

Moved Hotel_Booking_Confirmation.pdf to ./data/raw
Moved Hyderabad_Travel_Brochure (1).pdf to ./data/raw

--- Files in data/raw after moving ---
['Hotel_Booking_Confirmation.pdf', 'Hyderabad_Travel_Brochure (1).pdf']


In [3]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load: Point to the 'raw' folder
loader = DirectoryLoader('./data/raw', glob="*.pdf", loader_cls=PyPDFLoader)
docs = loader.load()

# 2. Chunk: Split into smaller pieces for better logic processing
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

# 3. Embed: Use Gemini to turn text into math vectors
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 4. Store: Save the index to 'processed' for Phase 1 persistence[cite: 1]
vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("./data/processed/faiss_index")

print(f"✅ Knowledge base built with {len(chunks)} document chunks.")

✅ Knowledge base built with 11 document chunks.


In [4]:
print("\n--- Document Chunks Inspection ---")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}:\n{chunk.page_content}\n---\n")

print(f"Total chunks: {len(chunks)}")


--- Document Chunks Inspection ---
Chunk 0:
HOTEL BOOKING CONFIRMATION
Field
Details
Confirmation #
HYD-2026-456789
Hotel Name
The Grand Hyderabad Palace
Address
Banjara Hills, Hyderabad - 500034
Phone
+91-40-67890123
Email
bookings@grandhyd.com
Website
www.grandhyd.com
Guest Name
John David
Email
john.david@email.com
Phone
9876543210
Check-in Date
June 15, 2026
Check-in Time
2:00 PM (STRICT - Late arrivals after 11:00 PM incur Rs. 50% room charge)
Check-out Date
June 20, 2026
Check-out Time
11:00 AM SHARP (Late checkout before 2 PM: Rs. 1,500 | 2-4 PM: Rs. 3,000 | After 4 PM: Full night charge)
Number of Nights
5 nights
Room Type
Deluxe Double Room (with city view)
Room Number
TBD at check-in
Room Rate
Rs. 8,500/night
Number of Rooms
1
Subtotal
Rs. 42,500
GST (18%)
Rs. 7,650
Resort Fee
Rs. 500/night (Rs. 2,500)
Total Amount
Rs. 52,650
Amount Paid
Rs. 52,650 (100% advance)
Balance Due
Rs. 0
HOTEL POLICIES & FINE PRINT
Critical Check-in/Check-out Times:
---

Chunk 1:
Balance Due
Rs. 0


In [5]:
import google.generativeai as genai

print("Available Embedding Models:")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(m.name)

Available Embedding Models:
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [6]:
import os
import langchain

print("--- Inspecting LangChain Package Contents ---")

try:
    langchain_path = langchain.__path__[0]
    print(f"Checking contents of: {langchain_path}")

    # List immediate contents
    package_contents = os.listdir(langchain_path)
    print(f"Immediate contents: {package_contents}")

    # Check for 'chains' module/directory
    chains_path = os.path.join(langchain_path, 'chains')
    if os.path.exists(chains_path):
        if os.path.isdir(chains_path):
            print(f"'chains' directory found at: {chains_path}")
            print(f"Contents of 'chains' directory: {os.listdir(chains_path)}")
        else:
            print(f"'chains.py' file found at: {chains_path}")
    else:
        print("'chains' module/directory NOT found within langchain package.")

except Exception as e:
    print(f"Error inspecting langchain package: {e}")

print("--- End of Package Inspection ---")

--- Inspecting LangChain Package Contents ---
Checking contents of: /usr/local/lib/python3.12/dist-packages/langchain
Immediate contents: ['sql_database.py', 'input.py', 'agents', 'chat_loaders', 'env.py', 'cache.py', 'text_splitter.py', 'python.py', 'evaluation', 'storage', '_api', 'example_generator.py', 'adapters', 'graphs', 'globals', 'load', 'docstore', '__pycache__', 'chat_models', 'formatting.py', 'prompts', 'utils', 'py.typed', 'document_loaders', 'vectorstores', 'base_language.py', '__init__.py', 'output_parsers', 'embeddings', 'document_transformers', 'memory', 'schema', 'smith', 'llms', 'callbacks', 'runnables', 'serpapi.py', 'pydantic_v1', 'hub.py', 'indexes', 'retrievers', 'model_laboratory.py', 'utilities', 'requests.py', 'tools', 'chains']
'chains' directory found at: /usr/local/lib/python3.12/dist-packages/langchain/chains
Contents of 'chains' directory: ['query_constructor', 'llm_summarization_checker', 'ernie_functions', 'graph_qa', 'structured_output', 'natbot', 'llm

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0) # Changed model to gemini-2.5-flash

# Define the Persona (The "Intelligent Response" logic)
system_prompt = (
    "You are a professional AI Travel Concierge. "
    "Use the provided context to answer the user's question accurately. "
    "If the answer is not in the context, politely say you don't know. "
    "\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Create the Retrieval Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(vector_store.as_retriever(), combine_docs_chain)

# TEST: Run a query to verify the milestone
response = rag_chain.invoke({"input": "What is the check-in time and the refund policy?"})

print("\n--- AI TRAVEL CONCIERGE RESPONSE ---")
print(response) # Print the entire response object for debugging


--- AI TRAVEL CONCIERGE RESPONSE ---
{'input': 'What is the check-in time and the refund policy?', 'context': [Document(page_content='Balance Due\nRs. 0\nHOTEL POLICIES & FINE PRINT\nCritical Check-in/Check-out Times:\nCheck-in: 2:00 PM (Early check-in at 6:00 AM available for Rs. 2,500 if room available)\nCheck-out: 11:00 AM SHARP (Non-negotiable - Late checkout charges: Before 2 PM: Rs. 1,500 | 2-4 PM: Rs. 3,000 | After 4 PM:\n100% room charge for extra night)\nLate Arrival: Arrivals after 11:00 PM incur 50% room charge even if room is held. Cancellation after 11:00 PM = full charge.', metadata={'source': 'data/raw/Hotel_Booking_Confirmation.pdf', 'page': 0}), Document(page_content='Cancellation Policy: Cancellations up to 7 days before check-in = 90% refund (10% admin fee). 3-6 days = 50% refund. 1-2\ndays = 0% refund. No-show = 100% forfeiture.\nRoom Details: Room number will be assigned at check-in. Early room assignment not possible before 12:00 PM. Room rates for\n2 guests. Add

In [8]:
import google.generativeai as genai

print("Available Generative Models:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Available Generative Models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robot